# Module 3: Hybrid Retrieval + Reranking

## What’s new?

We are improving retrieval by:

- Combining keyword search (BM25)
- Combining semantic search (vector)
- Reranking results

## Why?

Single retrieval methods are not sufficient for enterprise RAG.

Hybrid retrieval improves:
- recall
- precision
- robustness

## Architecture
    User Query
    ↓
    BM25 Retriever ───┐
                        ├── Merge → Rerank → Top-K → LLM
    Vector Retriever ─┘

In [15]:
%pip install -qU rank_bm25 langchain_community langchain langchain-openai langchain-chroma langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


## Step 1: Init Env,  Copy Docs and Chunks from Module 2

In [17]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

In [18]:
# Initialize environment variables for Azure OpenAI
api_key = (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
embedding_deployment = (os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME") or "").strip()
endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip()
llm_deployment = (
    os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    or ""
).strip()


# Create embeddings client based on endpoint type
# Foundry project endpoint -> OpenAI-compatible v1 endpoint.
if "services.ai.azure.com" in endpoint and "/api/projects/" in endpoint:
    resource_root = endpoint.split("/api/projects/")[0]
    openai_endpoint = f"{resource_root}/openai/v1"
    embed_model = OpenAIEmbeddings(
        model=embedding_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
    )
    embed_model.embed_query("endpoint-check")
    print("Embeddings client initialized OpenAI-compatible endpoint.")

    llm = ChatOpenAI(
        model=llm_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print("LLM initialized via OpenAI-compatible endpoint.")
else:
    api_version = (os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01").strip()
    embed_model = AzureOpenAIEmbeddings(
        model=embedding_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        openai_api_version=api_version,
    )
    embed_model.embed_query("endpoint-check")
    print(f"Embeddings client initialized via Azure OpenAI endpoint (api_version={api_version}).")

    llm = AzureChatOpenAI(
        azure_deployment=llm_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print(f"LLM initialized via Azure OpenAI endpoint (api_version={api_version}).")
    
    
   

Embeddings client initialized OpenAI-compatible endpoint.
LLM initialized via OpenAI-compatible endpoint.


In [9]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain helps build RAG systems using LLMs and retrievers.",
        metadata={"source": "langchain.pdf", "page": 1}
    ),
    Document(
        page_content="LangGraph enables multi-step workflows and agent orchestration.",
        metadata={"source": "langgraph.pdf", "page": 2}
    ),
    Document(
        page_content="LangSmith provides observability and evaluation for LLM systems.",
        metadata={"source": "langsmith.pdf", "page": 3}
    ),
]

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 3


## Step 2: Create BM25 Retriever

BM25 is keyword-based retrieval.

In [19]:
from langchain_community.retrievers import BM25Retriever

In [20]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

## Step 3: Vector Retriever (from Module 2)

In [13]:
from langchain_chroma import Chroma

In [21]:
# Embeddings and Persistent DB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="./chroma_db"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [30]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata['source']} (Page {doc.metadata.get('page', 'N/A')})\n{doc.page_content}"
        for doc in docs
    )

In [29]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Include source citations in your answer.

Context:
{context}

Question:
{question}
""")

## Step 3: Hybrid Retrieval

We combine results from:
- BM25 (keyword)
- Vector (semantic)

In [27]:
def hybrid_retrieve(query):
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = retriever.invoke(query)
    
    # Combine and deduplicate
    combined = bm25_docs + vector_docs
    
    unique_docs = {doc.page_content: doc for doc in combined}
    
    return list(unique_docs.values())

## Step 4: Simple Reranking (Simulation)

In production:
- Use cross-encoder models

Here:
- We simulate reranking using keyword overlap

In [24]:
def simple_reranker(query, docs):
    def score(doc):
        return sum(word in doc.page_content.lower() for word in query.lower().split())
    
    ranked = sorted(docs, key=score, reverse=True)
    return ranked[:3]

## Step 5: Full Pipeline

In [25]:
def ask_hybrid_rag(query):
    docs = hybrid_retrieve(query)
    reranked_docs = simple_reranker(query, docs)
    
    context = format_docs(reranked_docs)
    
    final_prompt = prompt.invoke({
        "context": context,
        "question": query
    })
    
    response = llm.invoke(final_prompt)
    
    return {
        "answer": response.content,
        "docs": reranked_docs
    }

In [32]:
result = ask_hybrid_rag("What does LangGraph do?")

print("Answer:\n", result["answer"])

print("\nDocuments Used:")
for d in result["docs"]:
    print(d.metadata)

Answer:
 LangGraph enables **multi-step workflows and agent orchestration**. 【langgraph.pdf, Page 2】

Documents Used:
{'source': 'langgraph.pdf', 'page': 2}
{'source': 'langsmith.pdf', 'page': 3}
{'page': 1, 'source': 'langchain.pdf'}
